# 第 27 章：套息、防御与低波动（Carry / Defensive / Low Volatility）

本 Notebook 复现正文第 8–11 节：外汇远期、零息债与商品曲线的 carry，换月时拼接价格与可交易收益的差，BAB 的权重、杠杆与收缩，构造的外汇 carry 世界，以及 French 10 行业上的低 beta 与 BAB。

- 手算例、外汇世界（种子 20261114）、商品世界（种子 20261115）是教学构造，已知数据生成过程；French 行业与 FF3 是 2026-10-05 冻结的已修订历史快照（10 个行业组合，不是个股，不是成交）。
- 核心函数来自 `src/quant_course/carry.py`、`low_risk.py`、`factor_research.py`；章节 CLI 在 `code/ch27_carry_lowrisk.py`。本 Notebook 不写文件，须从第一个单元顺序运行。

**离线构造数据教学示例。** French 数据未随包提供，相关使用权与实证执行授权尚未确立，因此本 Notebook 不运行真实数据分支。构造示例与静态教学图均不代表完整实证复现。


In [ ]:
from pathlib import Path
import sys, os, uuid
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start inside the reader package tree")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch27_carry_lowrisk_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.chdir(ROOT)
# French branches remain disabled until data rights and execution authorization are resolved.
RUN_FRENCH = False
print("CONSTRUCTED teaching route; French market branches SKIPPED; outputs:", RUN_DIR)

import sys
from pathlib import Path

sys.path.insert(0, str(ROOT / "src"))
print("interpreter:", sys.executable)
print("project root:", ROOT)
import importlib.util


def load_code_module(filename: str):
    """Import a chapter CLI from code/ so the notebook calls exactly the functions the CLI runs."""
    spec = importlib.util.spec_from_file_location(Path(filename).stem, ROOT / "code" / filename)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

## 1. 外汇远期、零息债、商品曲线（8.1–8.3）

In [ ]:
import numpy as np
import pandas as pd
from quant_course.carry import commodity_curve_carry, forward_premium, fx_carry_from_rates, futures_carry, roll_down_approximation, zero_coupon_carry

s, r_local, r_foreign = 100.0, 0.02, 0.06
f = s * (1 + r_local) / (1 + r_foreign)
fx = {"forward": f, "carry (prices)": futures_carry(s, f), "carry (rates)": fx_carry_from_rates(r_local, r_foreign),
      "forward premium": forward_premium(s, f), "carry, inverted quote": futures_carry(1 / s, 1 / f),
      "forward return if spot = 96": (96.0 - f) / f, "deposit excess if spot = 96": (96.0 / s * (1 + r_foreign) - (1 + r_local)) / (1 + r_local)}
assert abs(fx["carry (prices)"] - fx["carry (rates)"]) < 1e-12 and fx["carry, inverted quote"] < 0
pd.Series(fx).round(6)

In [ ]:
z = zero_coupon_carry([1, 2, 3, 5], [0.02, 0.03, 0.035, 0.04], 3.0, 1.0, financing_rate=0.02)
z["duration approximation of roll-down"] = roll_down_approximation(2 / 1.03, 0.035, 0.03)
assert abs((1 + z["yield_component"]) * (1 + z["roll_down_component"]) - 1 - z["total"]) < 1e-12
print("commodity carry per month: contango", round(commodity_curve_carry(100, 101, 1, 2), 5), "| backwardation", round(commodity_curve_carry(100, 98, 1, 2), 5))
pd.Series(z).round(6)

## 2. 换月：拼接价格与可交易收益（8.4）

In [ ]:
from quant_course.carry import held_contract_returns, naive_splice_returns

dates = pd.period_range("2026-01", periods=4, freq="M")
prices = pd.DataFrame({"A": [102.0, 101.0, 100.0, np.nan], "B": [np.nan, np.nan, 103.0, 102.0]}, index=dates)
chosen = pd.Series(["A", "A", "B", "B"], index=dates)
table = pd.DataFrame({"tradable": held_contract_returns(prices, chosen), "naive splice": naive_splice_returns(prices, chosen)}).iloc[:3]
table["gap"] = table["naive splice"] - table["tradable"]
assert abs(table.loc[dates[1], "gap"] - 3 / 101) < 1e-12 and abs((1 + table["naive splice"]).prod() - 1) < 1e-12
print("cumulative: tradable", round((1 + table["tradable"]).prod() - 1, 5), "| naive", round((1 + table["naive splice"]).prod() - 1, 5))
table.round(5)

## 3. BAB：权重、杠杆、收益、中性与收缩（8.5、8.7、6.9）

In [ ]:
from quant_course.low_risk import bab_returns, bab_weights, vasicek_shrink

beta = pd.Series([0.6, 0.9, 1.1, 1.4], index=list("ABCD"))
wl, wh = bab_weights(beta)
bl, bh = float(wl @ beta), float(wh @ beta)
r = pd.Series([0.012, 0.015, 0.010, 0.020], index=beta.index)
idx = pd.period_range("2026-01", periods=2, freq="M")
betas = pd.DataFrame([beta.to_numpy()] * 2, index=idx, columns=beta.index)
out = bab_returns(pd.DataFrame([r.to_numpy()] * 2, index=idx, columns=beta.index), betas, borrow_spread=0.002)
assert abs(bl - 0.675) < 1e-12 and abs(bh - 1.325) < 1e-12
print("w_low", wl.tolist(), "w_high", wh.tolist(), "| leg betas", round(bl, 4), round(bh, 4), "| leverage", round(1 / bl, 4), round(1 / bh, 4))
out.iloc[[0]].round(6)

In [ ]:
g = lambda x, w: x / (w * x + 1 - w)
pd.DataFrame({w: {"long leg exposure": g(0.7, w), "short leg exposure": g(1.3, w), "BAB beta": g(0.7, w) - g(1.3, w)} for w in (1.0, 0.8, 0.6)}).round(4)

In [ ]:
# noisy betas: realised BAB beta against the shrinkage weight (chapter 6.9, exercise 27-P03)
from quant_course.low_risk import bab_weights
rows = {}
for w in (1.0, 0.8, 0.5, 0.2):
    vals = []
    for k in range(100):
        rng = np.random.default_rng(1000 + k)
        true = rng.normal(1.0, 0.25, 500)
        est = true + rng.normal(0.0, 0.25, 500)
        scaled = pd.Series(w * est + (1 - w))
        a, b = bab_weights(scaled)
        vals.append((a * true).sum() / (a * scaled).sum() - (b * true).sum() / (b * scaled).sum())
    rows[w] = np.mean(vals)
print(pd.Series(rows).round(4).to_dict())
assert rows[1.0] > 0.3 and abs(rows[0.5]) < 0.05 and rows[0.2] < -0.15

## 4. 构造的外汇 carry 世界与 200 个新种子（10 节）

In [ ]:
ch27 = load_code_module("ch27_carry_lowrisk.py")
world = ch27.fx_world()
fx_res = ch27.fx_study(world)
show = {k: v for k, v in fx_res.items() if not k.startswith("_")}
assert abs(show["mean_portfolio_carry_pct"] - 0.51584) < 1e-5 and show["skew"] < 0 and show["mean_return_pct"] < show["mean_portfolio_carry_pct"]
pd.Series(show).round(4)

In [ ]:
sweep = ch27.fx_sweep(60)          # the CLI uses 200 worlds; 60 keeps the notebook quick
assert sweep["share_skew_negative"] == 1.0
pd.Series({k: v for k, v in sweep.items() if not isinstance(v, (dict, list))}).round(4)

## 5. 商品世界：可交易 = carry + 价格变动

In [ ]:
com = ch27.commodity_study(ch27.commodity_world())
print({k: round(v, 4) for k, v in com.items() if not k.startswith("_")})
assert com["decomposition_max_abs_error"] < 1e-12 and com["corr_gap_next_slope"] > 0.99

## 6. French 10 行业：低 beta 一半、BAB、收缩权重、成本

In [ ]:
if RUN_FRENCH:
    real = ch27.industry_study()
    halves = pd.DataFrame({k: {"mean %": v["mean_pct"], "t": v["t_iid"], "Sharpe": v["sharpe_annual"], "beta": v["regression_on_market"]["beta"], "alpha %": v["regression_on_market"]["alpha_pct"]}
                           for k, v in real["halves"].items()}).T
    assert halves.loc["low_beta", "beta"] < 1 < halves.loc["high_beta", "beta"]
    halves.round(3)
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")


In [ ]:
if RUN_FRENCH:
    b = real["bab"]
    print("BAB: mean %.4f%%/month, t %.2f, realised beta %.3f, months %d" % (b["mean_pct"], b["t_iid"], b["regression_on_market"]["beta"], b["months"]))
    variants = pd.DataFrame({k: {"mean %": v["mean_pct"], "t": v["t_iid"], "beta": v["regression_on_market"]["beta"]} for k, v in real["bab_shrink_variants"].items()}).T
    assert abs(real["bab_shrink_variants"]["1.0"]["regression_on_market"]["beta"]) < abs(b["regression_on_market"]["beta"])
    variants.round(3)
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")


In [ ]:
if RUN_FRENCH:
    pd.DataFrame({"by period": {k: v["mean_pct"] for k, v in real["bab_by_period"].items()}}).join(pd.DataFrame({"t": {k: v["t_iid"] for k, v in real["bab_by_period"].items()}})).round(3)
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")


In [ ]:
if RUN_FRENCH:
    print("max drawdown:", round(real["bab_max_drawdown"]["value"], 3), real["bab_max_drawdown"]["trough_return_month"], "| corr", {k: round(v, 3) for k, v in real["bab_corr"].items()})
    pd.DataFrame(real["bab_worst_10"]).round(2)
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")


In [ ]:
print("SKIPPED historical figure gallery; constructed calculations above are freshly computed; French historical images are not reproduced")


## 7. 自己试一试

1. 把外汇世界的 `gamma` 改成 0.0 与 1.0（`ch27.fx_world(gamma=...)`），看平均收益与平均组合 carry 的比值如何变化，解释 6.5 节。
2. 把 `ch27.fx_world(crash_prob=0.0)`，对比偏度与最大回撤，说明左尾来自哪一部分。
3. 把 `industry_study()` 里的收缩权重 0.6 改成 0.9，看 BAB 的真实 beta 是否接近零，并写下这是不是一个干净的检验。